In [1]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import os
import logging
logging.getLogger('matplotlib.font_manager').disabled = True
plt.style.use("https://raw.githubusercontent.com/NeuromatchAcademy/course-content/main/nma.mplstyle")

#Spectrogram Specific Imports
import pandas as pd
from scipy import signal

In [2]:
# Subcritical Hopf Imports

from subcritical_hopf import *

def classify_trajectory(r):
    """
    Classifies the trajectory based on its final state:
    - Aqua Green (#ffb300): Ends at the stable fixed point (FP)
    - Deep Violet (#490648): Stays on the stable limit cycle (LC)
    """
    tol = 1e-2  # Tolerance for determining stability
    final_r = r[-1]
    
    if final_r < 0.6:  # Close to the stable FP at the origin
        return '#9467bd'
    elif final_r > 0.7:  # Threshold for stable LC (adjustable)
        return '#1f77b4'

def plot_colored_subcritical_hopf(mu, omega, b, r_inits, theta_inits, dt, T, arrow_stop_time=None):
    """
    Plots a subcritical Hopf bifurcation phase portrait with color-coded trajectories and directional arrows.

    Parameters:
        mu, omega, b: model parameters
        r_inits, theta_inits: arrays of initial conditions
        dt: timestep
        T: total simulation time
        arrow_stop_time: optional float, in seconds. Arrows will stop forming after this time.
    """
    plt.figure(figsize=(7, 7))

    arrow_every = 2000  # Plot an arrow every 2000 time steps

    # Default: plot arrows for 25% of the simulation unless arrow_stop_time is given
    max_arrow_time = int((arrow_stop_time / dt) if arrow_stop_time else 0.25 * (T / dt))

    for r_init in r_inits:
        for theta_init in theta_inits:
            r, theta = subcritical_hopf(mu, omega, b, r_init, theta_init, dt, T)
            r = np.abs(r)
            x = r * np.cos(theta)
            y = r * np.sin(theta)

            color = classify_trajectory(r)

            # Plot trajectory
            plt.plot(x, y, color=color, alpha=0.8)

            # Add arrows along trajectory (up to max_arrow_time)
            for i in range(0, min(len(x) - 1, max_arrow_time), arrow_every):
                dx = x[i + 1] - x[i]
                dy = y[i + 1] - y[i]
                plt.arrow(
                    x[i], y[i], dx, dy,
                    head_width=0.025, head_length=0.05,
                    fc=color, ec=color, linewidth=0.5,
                    alpha=0.9, length_includes_head=True
                )

    # Set labels, grid, and limits
    plt.xlabel(r'$r_E$')
    plt.ylabel(r'$r_I$')
    plt.grid()
    plt.xlim(-1.2, 1.2)
    plt.ylim(-1.2, 1.2)

In [3]:
# PANEL A Phase Plane: mu = - 0.245

# Output directory
output_dir = "/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D"
os.makedirs(output_dir, exist_ok=True)

# Parameters
mu_values = [-0.245]
r_inits = [0.6, 0.7]
theta_inits = [0]
omega = 1
b = 2
dt = 0.001
T = 200

for mu in mu_values:
    
    plot_colored_subcritical_hopf(mu, omega, b, r_inits, theta_inits, dt, T, arrow_stop_time=20.0)

    roots = find_fixed_points(mu)
    print(f"Roots for mu={mu}: {roots}")

    r_ulc = roots[0]
    thetas = np.linspace(0, 2 * np.pi, 300)
    x_ulc = r_ulc * np.cos(thetas)
    y_ulc = r_ulc * np.sin(thetas)

    # Plot the unstable limit cycle (ULC) + stable fixed point (SFP)
    plt.plot(x_ulc, y_ulc, linestyle='--', color='#ff7f0e', alpha=1, linewidth=2)
    plt.plot(0, 0, 'o', color='#9467bd', markersize=6, zorder=5)

    # Add LEGEND (just once)
    if mu == -0.245:
        # Proxy artists for legend
        fp_patch = plt.Line2D([0], [0], marker='o', color='#9467bd', linestyle='None', markersize=5, label='Stable Fixed Point')
        lc_patch = plt.Line2D([0], [0], color='#1f77b4', linewidth=1.5, label='Stable Limit Cycle')
        ulc_patch = plt.Line2D([0], [0], linestyle='--', color='#ff7f0e', linewidth=1.5, label='Unstable Limit Cycle')

        plt.legend(handles=[fp_patch, lc_patch, ulc_patch], loc='upper right', frameon=True, fontsize=18, handlelength=2, handletextpad=0.8)
        #plt.legend(handles=[fp_patch, lc_patch, ulc_patch], loc='upper right', frameon=True, fontsize = 18)

    subcritical_hopf_filename = os.path.join(output_dir, f"r={r_inits}_theta={theta_inits}_mu={mu}_T={T}.png")
    plt.savefig(subcritical_hopf_filename, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Run for mu={mu}_dt={dt}_T={T}_b={b}_omega={omega} completed. Saved in {output_dir}")

Roots for mu=-0.245: [0.6552017413601289, 0.7554539549957063]
Run for mu=-0.245_dt=0.001_T=200_b=2_omega=1 completed. Saved in /Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D


In [4]:
# PANEL B Phase Plane: mu = 0.1

# Output directory
output_dir = "/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D"
os.makedirs(output_dir, exist_ok=True)

# Parameters
mu_values = [0.1]  # Stable FP regime
r_inits = [0.05]
theta_inits = [0]
omega = 1
b = 2
dt = 0.001
T = 200

for mu in mu_values:
    
    plot_colored_subcritical_hopf(mu, omega, b, r_inits, theta_inits, dt, T, arrow_stop_time= 170.0)

    # Optional: Show the stable fixed point at origin
    # plt.plot(0, 0, marker='o', color='purple', markersize=6, label='Stable FP')

    # DO NOT plot ULC — there is none in this regime

    subcritical_hopf_filename = os.path.join(output_dir, f"r={r_inits}_theta={theta_inits}_mu={mu}_T={T}.png")
    plt.savefig(subcritical_hopf_filename, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Run for mu={mu}_dt={dt}_T={T}_b={b}_omega={omega} completed. Saved in {output_dir}")

Run for mu=0.1_dt=0.001_T=200_b=2_omega=1 completed. Saved in /Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D


In [5]:
# PANEL C Phase Plane: mu = -0.255

# Output directory
output_dir = "/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D"
os.makedirs(output_dir, exist_ok=True)

# Parameters
mu_values = [-0.255]  # Stable FP regime
r_inits = [0.9]
theta_inits = [0]
omega = 1
b = 2
dt = 0.001
T = 200

for mu in mu_values:
    
    plot_colored_subcritical_hopf(mu, omega, b, r_inits, theta_inits, dt, T, arrow_stop_time=50.0)

    plt.plot(0, 0, 'o', color='#9467bd', markersize=6, zorder=5)

    # DO NOT plot ULC — there is none in this regime

    subcritical_hopf_filename = os.path.join(output_dir, f"r={r_inits}_theta={theta_inits}_mu={mu}_T={T}.png")
    plt.savefig(subcritical_hopf_filename, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Run for mu={mu}_dt={dt}_T={T}_b={b}_omega={omega} completed. Saved in {output_dir}")

Run for mu=-0.255_dt=0.001_T=200_b=2_omega=1 completed. Saved in /Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D


In [6]:
# PANEL A,B,C Activity Trace (UPDATED)

# PANEL A: mu = - 0.245
# PANEL B: mu = 0.1
# PANEL C: mu = -0.255

from matplotlib.patches import Patch

# Output path
output_dir = "/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D"
os.makedirs(output_dir, exist_ok=True)

# Global figure style
plt.rcParams['figure.figsize'] = [14, 6]
plt.rcParams['font.size'] = 14

# Subcritical Hopf parameters
mu = -0.255
omega = 1
b = 1
dt = 0.001
T = 200
T_steps = int(T / dt)
time = np.arange(0, T, dt)

# Perturbation parameters
perturb_times = [0, 50, 100]
perturb_duration = 1
r_targets = [0.3, 0.6, 0.9]
theta_reset = 0.0

# Initialize arrays
r = np.zeros(T_steps)
theta = np.zeros(T_steps)
x = np.zeros(T_steps)

# Initial conditions
r[0] = 0.0
theta[0] = 0
x[0] = r[0] * np.cos(theta[0])

# Euler integration with perturbations
for t in range(1, T_steps):
    dr = mu * r[t - 1] + r[t - 1]**3 - r[t - 1]**5
    dtheta = omega + b * r[t - 1]**2

    r[t] = r[t - 1] + dr * dt
    theta[t] = theta[t - 1] + dtheta * dt

    if int(time[t]) in perturb_times:
        idx = perturb_times.index(int(time[t]))
        r[t] = r_targets[idx]
        theta[t] = theta_reset

    x[t] = r[t] * np.cos(theta[t])

# Plot
plt.figure(figsize=(14, 6))
plt.plot(time, x, label=r'$r_E$', color='green')

# Perturbation shading
perturb_colors = ['yellow', 'orange', 'red']
for i, t_pert in enumerate(perturb_times):
    plt.axvspan(t_pert, t_pert + perturb_duration, color=perturb_colors[i], alpha=0.4)

# Custom legend
legend_elements = [
    Patch(facecolor='yellow', edgecolor='yellow', alpha=0.4, label='Low Perturbation'),
    Patch(facecolor='orange', edgecolor='orange', alpha=0.4, label='Medium Perturbation'),
    Patch(facecolor='red', edgecolor='red', alpha=0.4, label='High Perturbation'),
    plt.Line2D([0], [0], color='green', label=r'$r_E$')
]

# Axis labels and ticks
plt.xlabel('Time (ms)', fontsize=30) #used to be 24 24 20 20 and legend @ 18
plt.ylabel('Activity', fontsize=30)
plt.yticks([-1, 0, 1], fontsize=24)
plt.xticks(fontsize=24)

# Legend placement
plt.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(0.66, 0.81), frameon=True, fontsize=20)

# Save
plot_path = os.path.join(output_dir, f"SyntheticPerturbation_rCos_rSin_b={b}_mu={mu}.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close()

In [7]:
# Wilson Cowan Imports

from wilson_cowan import *

In [8]:
# PANEL D Phase Plane (UPDATED)

# Define output directory
output_dir = "/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D"
os.makedirs(output_dir, exist_ok=True)  # Ensure directory exists

# Set simulation parameters
pars = default_pars(T=1000)
pars['I_ext_I'] = 19
pars['I_ext_E'] = 22
pars['wII'] = 0.5

# Auto-generate filename for saving
output_filename = "Panel_D_Phase_Plane"
plot_filename = os.path.join(output_dir, f"{output_filename}.png")
text_filename = os.path.join(output_dir, f"{output_filename}.txt")
 
plt.figure(figsize=(7, 7))

# Find and plot the correct fixed points (Color coded)
x_fp_1 = my_fp(pars, 0.1, 0.8)  # Stable oscillatory FP (purple)
if check_fp(pars, x_fp_1):
    plot_fp(x_fp_1, fp_index=1, position=(0, 0.04), rotation=0, color='#9467bd')

x_fp_2 = my_fp(pars, 0.5, 0.9)  # Dashed line boundary FP (dark grey)
if check_fp(pars, x_fp_2):
    plot_fp(x_fp_2, fp_index=2, position=(0, .02), rotation=0, color='#ff7f0e')

x_fp_3 = my_fp(pars, 0.8, 0.9)  # High activity stable FP (BLUE)
if check_fp(pars, x_fp_3):
    plot_fp(x_fp_3, fp_index=3, position=(0, .02), rotation=0, color='#1f77b4')

# purple, burgundy, greyish blue

# Plot the dividing line (Dashed line boundary in RED)
rE_vals = np.linspace(0, 1, 100)  # Generate x values
rI_vals = 0.8 * rE_vals + 0.53  # Compute corresponding y values
plt.plot(rE_vals, rI_vals, '--', color='#ff6f00', linewidth=3,)

# my_plot_trajectories parameters
start=0.05
stop=1.0
dx=0.167

from matplotlib.lines import Line2D

def my_plot_trajectories_with_arrows(pars, start=0.1, stop=1.0, dx=0.2, arrow_interval=10, arrow_max_time=None):
    """
    Plots full WC model trajectories, but adds arrows only up to arrow_max_time.

    Parameters:
    - arrow_interval: Steps between arrow placements
    - arrow_max_time: Max time (ms) up to which arrows are plotted. Trajectories are always plotted fully.
    """
    rE_vals = np.arange(start, stop, dx)
    rI_vals = np.arange(start, stop, dx)

    arrow_steps = int(arrow_max_time / pars['dt']) if arrow_max_time else None

    for rE_0 in rE_vals:
        for rI_0 in rI_vals:
            pars_copy = pars.copy()
            pars_copy.update({'rE_init': rE_0, 'rI_init': rI_0})

            rE, rI = simulate_wc(**pars_copy)

            # Choose color
            color = '#1f77b4' if rE[-1] > 0.8 else '#9467bd'

            # Plot full trajectory
            plt.plot(rE, rI, color=color, alpha=0.8, linewidth=1.0)

            # Determine arrow cutoff index
            max_index = arrow_steps if arrow_steps else len(rE)

            # Plot arrows only up to arrow_max_time
            for i in range(0, min(len(rE) - 1, max_index - 1), arrow_interval):
                dx_arrow = rE[i + 1] - rE[i]
                dy_arrow = rI[i + 1] - rI[i]
                plt.arrow(rE[i], rI[i], dx_arrow, dy_arrow,
                          head_width=0.01, head_length=0.015,
                          fc=color, ec=color, linewidth=0.1,
                          alpha=0.9, length_includes_head=True)

    plt.xlabel(r'$r_E$', fontsize=16)
    plt.ylabel(r'$r_I$', fontsize=16)
    plt.tick_params(axis="both", labelsize=16)

# Use the updated function
my_plot_trajectories_with_arrows(pars, start=start, stop=stop, dx=dx, arrow_interval=750, arrow_max_time=75)

plt.xlim(0, 1.05)
plt.ylim(0, 1.05)

legend_elements = [
    Line2D([0], [0], marker='o', color='#1f77b4', linestyle='None', markersize=6, label='Stable Normal FP'),
    Line2D([0], [0], marker='o', color='#9467bd', linestyle='None', markersize=6, label='Stable Oscillatory FP'),
    Line2D([0], [0], linestyle='--', color='#ff7f0e', linewidth=2, label='Unstable Equilibrium'),
]

plt.legend(handles=legend_elements, loc='best', fontsize=16, frameon=True)

# Save plot
plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
print(f"Plot saved to {plot_filename}")
plt.close()

# Save results to a text file
with open(text_filename, 'w') as f:
    f.write(f"Fixed Point Phase Plane Analysis - {output_filename}\n")
    f.write(f"External Inputs: I_ext_E={pars['I_ext_E']}, I_ext_I={pars['I_ext_I']}, wII={pars['wII']}\n")
    f.write(f"my_plot_trajectories: start={start}, stop={stop}, step={dx}\n")
    f.write(f"Fixed Points:\n")
    f.write(f"FP1 (Stable Oscillatory - Green): {x_fp_1}\n")
    f.write(f"FP2 (Dashed Line Boundary - Red): {x_fp_2}\n")
    f.write(f"FP3 (High Activity Stable - Blue): {x_fp_3}\n")
    f.write("\nUnstable Dividing Line Equation: rI = 0.8 * rE + 0.53\n")

print(f"Analysis results saved to {text_filename}")

Plot saved to /Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D/Panel_D_Phase_Plane.png
Analysis results saved to /Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing Panels A-D/Panel_D_Phase_Plane.txt


In [9]:
### --- 1️⃣ DEFINE CONSTANT PARAMETERS --- ###
tau = 5
D = 0.000000250  
dt = 0.1  
T = 20000  
T_steps = int(T / dt)  
nperseg = 256*3  
snapshot_duration = 500  

# Compute noise scaling
Ae = np.sqrt((D * tau) / 2 * (1 - np.exp(-2 * dt / tau)))

# Proper scientific notation for filenames
D_formatted = f"{D:.3e}"
output_filename = f"OU_Noise_Tau_D={D_formatted}_nperseg={nperseg}"

# Run 10 simulations, each in its own folder
for run in range(1, 2):
    output_dir = f"/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing_Spectrogram_2/tau {tau} - D {D} - run {run}"
    os.makedirs(output_dir, exist_ok=True)

    ### --- 2️⃣ GENERATE OU NOISE --- ###
    ou_rE = np.zeros(T_steps)
    ou_rI = np.zeros(T_steps)
    
    for t in range(1, T_steps):
        dW_rE, dW_rI = np.random.randn(), np.random.randn()
        ou_rE[t] = ou_rE[t - 1] * np.exp(-dt / tau) + Ae * dW_rE
        ou_rI[t] = ou_rI[t - 1] * np.exp(-dt / tau) + Ae * dW_rI
    
    ### --- 3️⃣ RUN WILSON-COWAN MODEL WITH OU NOISE --- ###
    pars = default_pars(T=T)  
    pars.update({'I_ext_I': 19, 'I_ext_E': 22, 'wII': 0.5, 'rE_init': 0.1, 'rI_init': 0.8})
    
    rE, rI = np.zeros(T_steps), np.zeros(T_steps)
    rE[0], rI[0] = pars['rE_init'], pars['rI_init']
    
    for t in range(1, T_steps):
        dE, dI = EIderivs(rE[t - 1], rI[t - 1], **pars)
    
        # Noise added to the dynamics, before scaling by dt
        noisy_dE = dE + ou_rE[t]
        noisy_dI = dI + ou_rI[t]
    
        # Euler integration
        rE[t] = rE[t - 1] + noisy_dE * dt
        rI[t] = rI[t - 1] + noisy_dI * dt

    # Save rE and rI time series data to a CSV file
    time_series_data = pd.DataFrame({
        'Time': np.arange(0, T, dt),
        'rE': rE,
        'rI': rI
    })
    time_series_filename = os.path.join(output_dir, f'time_series_run{run}.csv')
    time_series_data.to_csv(time_series_filename, index=False)

    # Generate snapshots every 500 ms
    for start_time in range(0, T, snapshot_duration):
        end_time = start_time + snapshot_duration
        start_idx, end_idx = int(start_time / dt), int(end_time / dt)

        ### --- 4️⃣ SPECTROGRAM ANALYSIS SNAPSHOT --- ###
        fs = 1000 / dt  
        f_e, t_e, Sxx_e = signal.spectrogram(rE[start_idx:end_idx], fs, nperseg=nperseg)
        t_e_shifted = np.linspace(start_time / 1000, end_time / 1000, len(t_e)) 

        # **Compute Summed Gamma Power (30-80 Hz)**
        gamma_mask = (f_e >= 30) & (f_e <= 80)
        summed_gamma_power = np.sum(Sxx_e[gamma_mask, :], axis=0)

        # Create figure with adjusted spacing to avoid axis label overlap
        fig = plt.figure(figsize=(14, 6))
        gs = fig.add_gridspec(7, 32, hspace=0.5, wspace=2.2)
        
        # WC + OU Noise Plot (Columns 1-8, Rows 2-5)
        ax1 = fig.add_subplot(gs[1:6, 1:13])
        x_plot = time[start_idx:end_idx]
        rE_plot = rE[start_idx:end_idx]
        rI_plot = rI[start_idx:end_idx]
        
        # Plot rE
        ax1.plot(x_plot, rE_plot, color='green', label=r'$r_E$')
        ax1.set_ylabel('rE', color='green', fontsize=12)
        ax1.tick_params(axis='y', labelcolor='green')
        ax1.set_ylim(0, 0.3)
        
        # Plot rI on right axis
        ax1_right = ax1.twinx()
        ax1_right.plot(x_plot, rI_plot, color='red', alpha=0.7, label=r'$r_I$')
        ax1_right.set_ylabel('rI', color='red', fontsize=12)
        ax1_right.tick_params(axis='y', labelcolor='red')
        ax1_right.set_ylim(0.7, 0.9)
        
        # Make all spines black and visible for a full rectangular box
        for ax in [ax1, ax1_right]:
            for spine in ['left', 'right', 'bottom']: #'top']:
                ax.spines[spine].set_visible(True)
                ax.spines[spine].set_color('black')
                ax.spines[spine].set_linewidth(1)
        
        # Shared x-axis
        ax1.set_xlabel('Time (ms)', fontsize=14)
        
        # Spectrogram (Columns 11-17, Rows 1-4)
        ax2 = fig.add_subplot(gs[0:5, 18:30])  # shifted right by 2 columns
        img = ax2.pcolormesh(t_e_shifted, f_e, Sxx_e, shading='gouraud', cmap='viridis', vmin=0, vmax=Sxx_e.max())
        ax2.set_ylabel('Frequency (Hz)', fontsize=14)
        ax2.set_ylim(0, 100)
        ax2.set_xlim(start_time / 1000, end_time / 1000)
        ax2.tick_params(labelbottom=False)
        
        # Summed Gamma Power (Columns 11-17, Rows 5-6)
        ax3 = fig.add_subplot(gs[5:7, 18:30])  # shifted right by 2 columns
        ax3.plot(t_e_shifted, summed_gamma_power, color='b')
        ax3.set_xlabel('Time (s)', fontsize=14)
        ax3.set_ylabel('Summed Gamma\n(30-80 Hz) Power', fontsize=14)
        ax3.set_xlim(start_time / 1000, end_time / 1000)
        
        ax3.spines['right'].set_visible(True)
        ax3.spines['right'].set_color('black')
        ax3.spines['right'].set_linewidth(1)
        
        # Colorbar (Column 18, All Rows)
        ax4 = fig.add_subplot(gs[:, 31])  # adjusted for 32 total columns
        cbar = fig.colorbar(img, cax=ax4)
        cbar.set_label('Power Spectral Density')

        # Save the combined figure
        spectrogram_snapshot_filename = os.path.join(output_dir, f"Spectrogram_{output_filename}_{start_time}-{end_time}ms.png")
        plt.savefig(spectrogram_snapshot_filename, dpi=300, bbox_inches='tight')
        plt.close()

    print(f"Run {run} completed. Saved in {output_dir}")

/var/folders/f_/359fxh894qq1z634_tklv9fw0000gn/T/ipykernel_1306/1198870631.py:131: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.savefig(spectrogram_snapshot_filename, dpi=300, bbox_inches='tight')


Run 1 completed. Saved in /Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing_Spectrogram_2/tau 5 - D 2.5e-07 - run 1


In [10]:
# Output path
output_dir = "/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures"
os.makedirs(output_dir, exist_ok=True)

#change the csv file name after /Finalizing_Spectrogram/ according to which run the CSV file belongs to + time_series_run# (change #)

# Read the CSV file containing rE and rI time series data
csv_filename = "/Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Finalizing_Spectrogram/tau 5 - D 2.5e-07 - run 12/time_series_run12.csv"
time_series_data = pd.read_csv(csv_filename)

# Extract time, rE, and rI from the CSV
time = time_series_data['Time'].values
rE = time_series_data['rE'].values
rI = time_series_data['rI'].values

# Parameters for the spectrogram and summed gamma power calculation
nperseg = 256 * 3  # Number of points per segment for the spectrogram
snapshot_duration = 500  # Duration of each snapshot (in ms)

# Here, I can edit start_time and end_time to get JUST the 500ms snapshot I want. start_time = 10000 ms and end_time = start_time + snapshot_duration (500ms snapshot)
start_time = 8500  # Use this for your start time of analysis
end_time = start_time + snapshot_duration  # End time for spectrogram analysis

start_idx = int(start_time / 0.1)  # Index for the start time
end_idx = int(end_time / 0.1)  # Index for the end time

# Spectrogram analysis snapshot
fs = 1000 / 0.1  # Sampling frequency (1000 ms per second / time step in ms)
f_e, t_e, Sxx_e = signal.spectrogram(rE[start_idx:end_idx], fs, nperseg=nperseg)
t_e_shifted = np.linspace(start_time / 1000, end_time / 1000, len(t_e))

# **Compute Summed Gamma Power (30-80 Hz)**
gamma_mask = (f_e >= 30) & (f_e <= 80)
summed_gamma_power = np.sum(Sxx_e[gamma_mask, :], axis=0)

# Create figure with adjusted spacing to avoid axis label overlap
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(7, 32, hspace=0.5, wspace=2.2)

# WC + OU Noise Plot (Columns 1-8, Rows 2-5)
ax1 = fig.add_subplot(gs[1:6, 1:13])
x_plot = time[start_idx:end_idx]
rE_plot = rE[start_idx:end_idx]
rI_plot = rI[start_idx:end_idx]

# Plot rE
ax1.plot(x_plot, rE_plot, color='green', label=r'$r_E$')
ax1.set_ylabel(r'$r_E$', color='green', fontsize=12)
ax1.tick_params(axis='y', labelcolor='green')
ax1.set_ylim(0, 0.3)

# Plot rI on right axis
ax1_right = ax1.twinx()
ax1_right.plot(x_plot, rI_plot, color='red', alpha=0.7, label=r'$r_I$')
ax1_right.set_ylabel(r'$r_I$', color='red', fontsize=12)
ax1_right.tick_params(axis='y', labelcolor='red')
ax1_right.set_ylim(0.7, 0.9)

# Make all spines black and visible for a full rectangular box
for ax in [ax1, ax1_right]:
    for spine in ['left', 'right', 'bottom']: #'top']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('black')
        ax.spines[spine].set_linewidth(1)

# Shared x-axis
ax1.set_xlabel('Time (ms)', fontsize=14)

# Spectrogram (Columns 11-17, Rows 1-4)
ax2 = fig.add_subplot(gs[0:5, 18:30])  # shifted right by 2 columns
img = ax2.pcolormesh(t_e_shifted, f_e, Sxx_e, shading='gouraud', cmap='viridis', vmin=0, vmax=Sxx_e.max())
ax2.set_ylabel('Frequency (Hz)', fontsize=14)
ax2.set_ylim(0, 100)
ax2.set_xlim(start_time / 1000, end_time / 1000)
ax2.tick_params(labelbottom=False)

# Summed Gamma Power (Columns 11-17, Rows 5-6)
ax3 = fig.add_subplot(gs[5:7, 18:30])  # shifted right by 2 columns
ax3.plot(t_e_shifted, summed_gamma_power, color='b')
ax3.set_xlabel('Time (s)', fontsize=14)
ax3.set_ylabel('Summed Gamma\n(30-80 Hz) Power', fontsize=14)
ax3.set_xlim(start_time / 1000, end_time / 1000)

ax3.spines['right'].set_visible(True)
ax3.spines['right'].set_color('black')
ax3.spines['right'].set_linewidth(1)

# Colorbar (Column 18, All Rows)
ax4 = fig.add_subplot(gs[:, 31])  # adjusted for 32 total columns
cbar = fig.colorbar(img, cax=ax4)
cbar.set_label('Power Spectral Density')

# Save the combined figure
plot_path = os.path.join(output_dir, f"Regenerated_Spectrogram_{os.path.basename(csv_filename)}.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Plot saved at {plot_path}")

/var/folders/f_/359fxh894qq1z634_tklv9fw0000gn/T/ipykernel_1306/329960106.py:95: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.savefig(plot_path, dpi=300, bbox_inches='tight')


Plot saved at /Users/TarunShriram/Documents/UCONN/Rich Lab/Figures/Regenerated_Spectrogram_time_series_run12.csv.png
